# 12 - Pass 6: two-stage training, pairs first, then gold labels (one commit, GPU)

Same structure as notebooks 08 and 11: fetch the code from GitHub, run the tests, check the data on the CPU, then train two runs at a time (one per GPU).

**What Pass 6 asks** (vault: Experiment Plan, "Pass 6 plan"): Passes 4-5 mixed the generated pairs into every training step, and the model lost 10-16 points on Headlines at any weight. Here the pairs come **first and alone**, then the model is fine-tuned on the real labels only, so the real labels have the last word. Does that keep the Headlines score at A1's level while keeping some of the iSarcasmEval transfer?

| Arm | Stage 1 (720 steps) | Stage 2 (usual steps) |
|---|---|---|
| A18 | generated pairs only (exactly Pass 4's A15) | real labels only, plain BERT (like A1) |
| A19 | the same | real labels only, BERT + FGM (like A7) |

5 seeds each, at 10% and 1% labels = 20 runs, test scored, masked test sets scored. Baselines reused, not re-run: A1, A7 (Passes 1-2), A15, A10, A11 (Pass 4), A10-weighted, A16, A17 (Pass 5).

Stage 1 is the same 720 steps at both budgets, so its model should be **identical** to Pass 4's A15 at 10% for the same seed. Step 6 checks this: the stage-1 dev score must equal A15's.

**Before you run, in the right-hand panel:**
- **Input:** the same `gainsplit-processed` version as notebooks 10 and 11.
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**, so the output and log are kept.


## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about a minute)
The repo's tests train a tiny random BERT for a few steps on made-up data, including the pair arms, the pair batches and the explanation step. They prove the pieces fit together before real GPU time is spent. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the data on the CPU (pre-flight)
Checks every file against `manifest.json` and `manifest_pass3.json`, then loads the `unfiltered` pair set and checks it. Expected: 44,834 rows (22,417 pairs). Any problem raises an error here, before training starts.


In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass6"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN]()

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
from gainsplit.plans import PASS6_PRETRAIN_STEPS
print("stage-1 steps:", PASS6_PRETRAIN_STEPS)
display(preflight(configs, DATA_DIR))

## 4. The list of runs and the time estimate
`pass6` = 20 runs, seeds 0-4: A18 and A19 at 10% and at 1%.

Estimated run time (from notebooks 08 and 11): stage 1 about 2.3 min per run; stage 2 about 1.4 min (A18, 10%), 3.3 min (A19, 10%), 0.5-1 min (1%). Total about 75 run-minutes, so **about 40 minutes on two GPUs**, plus about 5 minutes for steps 1-3.


In [ ]:
for c in configs:
    print(f"{c.run_id:24s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} eps={c.fgm_epsilon:g} "
          f"pretrain_steps={c.pretrain_steps}")
print(len(configs), "runs")


## 5. Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`. If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## 6. First look (the full analysis runs later on the CPU)
1. **Stage-1 check:** each run's stage-1 dev macro-F1 must equal Pass 4's A15 (10%, same seed) exactly (difference 0). A non-zero difference means stage 1 is not the A15 model: report it before reading anything else.
2. Mean test macro-F1, cue-masked F1 and iSarcasmEval AUC (points) per arm and budget, next to the baselines read from the repo clone.

What to look for: is A18 close to A1 (88.2 at 10%, 80.4 at 1%) and A19 close to A7 (89.8 / 82.0)? Does iSarcasmEval AUC stay clearly above 68.7 (word count alone) or fall back towards 50?


In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for f in sorted(Path(OUT_DIR).glob("runs/*/result.json")):
    r = json.loads(f.read_text()); c = r["config"]
    rows.append({"arm": c["arm"], "budget": c["budget"], "seed": c["seed"],
                 "stage1_dev_f1": r["pretrain"]["dev"]["macro_f1"] * 100,
                 "test_f1": r["scores"]["test"]["macro_f1"] * 100,
                 "cuemask_f1": r["scores"]["test_cuemask"]["macro_f1"] * 100,
                 "isarc_auc": r["scores"]["isarc_test"]["roc_auc"] * 100, "minutes": r["minutes"]})
res = pd.DataFrame(rows)

# 1. stage-1 check against Pass 4's A15 at 10% (same seed)
a15 = {}
for f in sorted((Path(REPO_DIR) / "results" / "pass4" / "runs").glob("A15_p10_s*/result.json")):
    r = json.loads(f.read_text()); a15[r["config"]["seed"]] = r["scores"]["dev"]["macro_f1"] * 100
res["a15_dev_f1"] = res["seed"].map(a15)
res["stage1_diff"] = (res["stage1_dev_f1"] - res["a15_dev_f1"]).abs()
display(res[["arm", "budget", "seed", "stage1_dev_f1", "a15_dev_f1", "stage1_diff"]])
print("stage-1 check:", "PASS (all equal A15)" if (res["stage1_diff"] == 0).all() else "DIFFERENCE - report this first")

# 2. means per arm and budget
display(res.groupby(["budget", "arm"])[["test_f1", "cuemask_f1", "isarc_auc", "minutes"]].mean().round(3))

base = []
for p in ("pass1", "pass2b", "pass4", "pass5"):
    s = Path(REPO_DIR) / "results" / p / "summary.csv"
    if s.exists():
        base.append(pd.read_csv(s).assign(source=p))
if base:
    base = pd.concat(base, ignore_index=True)
    base["tag"] = base["tag"].fillna("")
    keep_rows = (base["budget"].isin([1, 10]) & (
        (base["arm"].isin(["A1", "A7", "A10", "A11", "A15", "A16", "A17"]) & base["tag"].eq(""))
        | (base["arm"].eq("A10") & base["tag"].eq("weighted"))))
    show = base[keep_rows].assign(arm=lambda d: d["arm"] + d["tag"].map(lambda t: f"-{t}" if t else ""))
    show = show.rename(columns={"test_macro_f1": "test_f1", "isarc_test_roc_auc": "isarc_auc"})
    display((show.groupby(["budget", "arm", "source"])[["test_f1", "isarc_auc"]].mean() * 100).round(3)
            .rename(columns=lambda c: c + " (baseline)"))


## 7. Download
Zips the results folder. A commit keeps it in the Output tab: download `results_<plan>.zip` and send it with the log.

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")